In [ ]:
! pip install pypdf
! pip install -U "mineru[all]"
! pip install google

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import drive
from pypdf import PdfReader, PdfWriter

from mineru.cli.common import read_fn, do_parse

In [ ]:
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "true"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_DEVICE"] = "cuda"

In [ ]:
drive.mount('/content/drive')
INPUT_DIR = "/content/drive/MyDrive/learningagentbooks"
OUTPUT_DIR = "/content/drive/MyDrive/learningagentmarkdown"

In [ ]:
def shorten_pdf(input_path, output_path, pages):
  reader = PdfReader(input_path)
  writer = PdfWriter()

  for page_num in range(pages):
    writer.add_page(reader.pages[page_num])

  with open(output_path, "wb") as f:
    writer.write(f)

def check_file_names():
    os.makedirs("/content/drive/MyDrive/learningagentbooks", exist_ok=True)
    os.makedirs("/content/drive/MyDrive/learningagentmarkdown", exist_ok=True)

    visible_files = os.listdir("/content/drive/MyDrive/learningagentbooks")
    visible_files = visible_files + os.listdir("/content/drive/MyDrive/learningagentmarkdown")
    if len(visible_files) == 0:
        print("The folder is EMPTY!")
    else:
        for file in visible_files:
            print(f"Found: {file}")

In [ ]:
def convert_with_mineru(file_name: str) -> str:
    """Convert a PDF to Markdown using MinerU.

    Uses the "pipeline" backend, which runs straight on the GPU via
    torch/paddle -- no Docker, no separate inference server to babysit.
    """
    input_path = os.path.join(INPUT_DIR, file_name)
    stem = Path(file_name).stem
    output_path = os.path.join(OUTPUT_DIR, stem + ".md")

    pdf_bytes = read_fn(input_path)

    do_parse(
        output_dir=OUTPUT_DIR,
        pdf_file_names=[stem],
        pdf_bytes_list=[pdf_bytes],
        p_lang_list=["en"],
        backend="pipeline",
        parse_method="ocr",
        formula_enable=True,
        table_enable=True,
        image_analysis=False,
    )

    nested_md_path = os.path.join(OUTPUT_DIR, stem, "ocr", f"{stem}.md")
    shutil.copyfile(nested_md_path, output_path)

    return output_path

In [ ]:
input_file_name = "test_snippet.pdf"
if __name__ == "__main__":
    os.makedirs(INPUT_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    convert_with_mineru(input_file_name)